<center><h1>Vikraman_Madhulika_HW3</h1></center>
<br>
<br>

DSCI 552 Homework 4
===================

Name: Madhulika Vikraman
<br>
Github Username: madhulikavikraman
<br>
USC ID: 2083047114

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
import os
from scipy.stats import bootstrap


Get the AReM Data Set

The data is in the /data/AReM folder, with individual folders inside /data/AReM/AReM path and has not been manually changed. I am handling the file with space separators and trailing commas in the load function.

-------------------------------------------------------------------------

### (b) Test and Train Data

In [2]:
def load_data(parent_folder_path, start_range, end_range):
    df_list = []
    for i in range(start_range, end_range + 1):
        file_path = f"{parent_folder_path}/dataset{i}.csv"  # only considers files in parent folder within given range
        data_rows = []
        with open(file_path, "r") as f:
            for line_number, line in enumerate(f):
                if line_number < 5:
                    continue
                line = line.strip()
                if not line:
                    continue
                line = line.replace(",", " ")   # i am dynamically handling space separators and extra comma
                line = " ".join(line.split())
                data_rows.append(line.split(" "))
        df = pd.DataFrame(data_rows).astype(float)
        df_list.append(df)
    return df_list

train_data = []
train_dataset_list = [
                    ("../data/AReM/AReM/bending1", 3, 7),
                    ("../data/AReM/AReM/bending2", 3, 6),
                    ("../data/AReM/AReM/cycling", 4, 15),
                    ("../data/AReM/AReM/lying", 4, 15),
                    ("../data/AReM/AReM/sitting", 4, 15),
                    ("../data/AReM/AReM/standing", 4, 15),
                    ("../data/AReM/AReM/walking", 4, 15)
                    ]
test_data = []
test_dataset_list = [
                    ("../data/AReM/AReM/bending1", 1, 2),
                    ("../data/AReM/AReM/bending2", 1, 2),
                    ("../data/AReM/AReM/cycling", 1, 3),
                    ("../data/AReM/AReM/lying", 1, 3),
                    ("../data/AReM/AReM/sitting", 1, 3),
                    ("../data/AReM/AReM/standing", 1, 3),
                    ("../data/AReM/AReM/walking", 1, 3)
                    ]

for path, i, j in train_dataset_list:
    train_data.extend(load_data(path, i, j))
for path, i, j in test_dataset_list:
    test_data.extend(load_data(path, i, j))

all_train_lists = train_data
train_df = pd.concat(all_train_lists, ignore_index=True)   # full training data
all_test_lists = test_data
test_df = pd.concat(all_test_lists, ignore_index=True)     # full testing data
print(train_df.shape)
print(test_df.shape)

(33119, 7)
(9120, 7)


-------------------------------------------------------------------------

### (c) Feature Extraction

#### i. Research

**Statistical measures that are usually used in time series classification:-**

Mean

Median

Mode

Variance

Standard Deviation

Range

Minimum

Maximum

Kurtosis

Skewness

Quantiles

Cross-Correlations between each dimension

Auto-correlation

-------------------------------------------------------------------------

#### ii. Extraction

In [3]:
train_df.head()

,0,1,2,3,4,5,6
0,0.0,42.00,0.71,21.25,0.43,30.00,0.00
1,250.0,41.50,0.50,20.25,1.48,31.25,1.09
2,500.0,41.50,0.50,14.25,1.92,33.00,0.00
3,750.0,40.75,0.83,15.75,0.43,33.00,0.00
4,1000.0,40.00,0.71,20.00,2.74,32.75,0.43


In [4]:
train_df.describe()

,0,1,2,3,4,5,6
count,33119.000000,33119.000000,33119.000000,33119.000000,33119.000000,33119.000000,33119.000000
mean,59876.400254,39.005285,1.510208,14.166349,1.505252,15.669678,1.644033
std,34641.049621,5.986658,2.082433,5.346573,1.648251,6.287338,1.646942
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,30000.000000,35.250000,0.430000,11.500000,0.430000,12.000000,0.470000
50%,60000.000000,39.670000,0.500000,14.750000,0.830000,15.670000,1.000000
75%,89875.000000,43.500000,1.920000,18.000000,2.290000,18.750000,2.380000
max,119750.000000,56.250000,17.240000,32.750000,11.420000,40.330000,13.610000


In [5]:
def load_data_instance(instance_path):
    data_rows = []
    with open(instance_path, "r") as f:
        for i, line in enumerate(f):
            if i < 5:
                continue
            line = line.strip()
            if not line:
                continue
            line = line.replace(",", " ")
            line = " ".join(line.split())
            data_rows.append(line.split(" "))
    df = pd.DataFrame(data_rows).astype(float)
    df = df.drop(df.columns[0], axis=1)  # drop time column
    return df

all_dataset_folder_list = [
                    ("../data/AReM/AReM/bending1"),
                    ("../data/AReM/AReM/bending2"),
                    ("../data/AReM/AReM/cycling"),
                    ("../data/AReM/AReM/lying"),
                    ("../data/AReM/AReM/sitting"),
                    ("../data/AReM/AReM/standing"),
                    ("../data/AReM/AReM/walking")
                    ]

def extract_features(all_dataset_folder_list):
    columns = ['Instance']
    for i in range(1, 7):
        columns.extend([f'Minimum{i}', f'Maximum{i}', f'Mean{i}', f'Median{i}', f'StdDev{i}', f'First Quartile{i}', f'Third Quartile{i}'])
    c = 1
    summary_df = pd.DataFrame(columns=columns)
    for path in all_dataset_folder_list:
        for filename in os.listdir(path):
            if filename.endswith(".csv"):
                instance_path = os.path.join(path, filename)
                inst_df = load_data_instance(instance_path)
                row = [c]
                for col in inst_df.columns:
                    row.extend([
                        inst_df[col].min(),
                        inst_df[col].max(),
                        inst_df[col].mean(),
                        inst_df[col].median(),
                        inst_df[col].std(),
                        inst_df[col].quantile(0.25),
                        inst_df[col].quantile(0.75)
                    ])
                summary_df.loc[len(summary_df)] = row
                c += 1
    return summary_df

feature_df = extract_features(all_dataset_folder_list)
display(feature_df)

,Instance,Minimum1,Maximum1,Mean1,Median1,StdDev1,First Quartile1,Third Quartile1,Minimum2,Maximum2,...,StdDev5,First Quartile5,Third Quartile5,Minimum6,Maximum6,Mean6,Median6,StdDev6,First Quartile6,Third Quartile6
0,1.0,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
1,2.0,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
2,3.0,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
3,4.0,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
4,5.0,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84.0,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
84,85.0,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
85,86.0,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200
86,87.0,18.33,45.75,34.599875,35.125,4.731790,31.50,38.0000,0.0,15.37,...,2.905688,14.0000,18.25,0.0,8.86,3.289542,3.015,1.680170,2.12,4.2600


-------------------------------------------------------------------------

#### iii. Standard Deviation

In [9]:
columns = ['Instance']
for i in range(1, 7):
    columns.extend([f'Minimum{i}', f'Maximum{i}', f'Mean{i}', f'Median{i}', f'StdDev{i}', f'First Quartile{i}', f'Third Quartile{i}'])
print('STANDARD DEVIATION OF EACH FEATURE:')
print(feature_df.loc[:, columns[1:]].std(axis=0))

STANDARD DEVIATION OF EACH FEATURE:
Minimum1           9.569975
Maximum1           4.394362
Mean1              5.335718
Median1            5.440054
StdDev1            1.772153
First Quartile1    6.153590
Third Quartile1    5.138925
Minimum2           0.000000
Maximum2           5.062729
Mean2              1.574164
Median2            1.412244
StdDev2            0.884105
First Quartile2    0.946386
Third Quartile2    2.125266
Minimum3           2.956462
Maximum3           4.875137
Mean3              4.008380
Median3            4.036396
StdDev3            0.946710
First Quartile3    4.220658
Third Quartile3    4.171628
Minimum4           0.000000
Maximum4           2.183625
Mean4              1.166114
Median4            1.145586
StdDev4            0.458242
First Quartile4    0.843620
Third Quartile4    1.552504
Minimum5           6.124001
Maximum5           5.741238
Mean5              5.675593
Median5            5.813782
StdDev5            1.024898
First Quartile5    6.096465
Third Quarti

In [7]:
data = feature_df.loc[:, columns[1:]]
data

,Minimum1,Maximum1,Mean1,Median1,StdDev1,First Quartile1,Third Quartile1,Minimum2,Maximum2,Mean2,...,StdDev5,First Quartile5,Third Quartile5,Minimum6,Maximum6,Mean6,Median6,StdDev6,First Quartile6,Third Quartile6
0,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
1,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
2,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
3,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
4,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,4.223792,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
84,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,4.115750,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
85,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,4.396958,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200
86,18.33,45.75,34.599875,35.125,4.731790,31.50,38.0000,0.0,15.37,4.398833,...,2.905688,14.0000,18.25,0.0,8.86,3.289542,3.015,1.680170,2.12,4.2600


In [8]:
ci_df = pd.DataFrame(columns=['Feature', 'Lower Bound', 'Upper Bound'])
for col in data.columns:
    colvals = data[col].to_numpy()
    bootstrap_ci = bootstrap((colvals,), np.std, confidence_level=0.9, random_state=1, method='percentile')
    ci_df.loc[len(ci_df)] = [col, bootstrap_ci.confidence_interval[0], bootstrap_ci.confidence_interval[1]]
display(pd.DataFrame(ci_df))

,Feature,Lower Bound,Upper Bound
0,Minimum1,8.208212,10.718897
1,Maximum1,3.302124,5.277910
2,Mean1,4.683108,5.864954
3,Median1,4.774987,5.983471
4,StdDev1,1.561586,1.930514
5,First Quartile1,5.548767,6.619312
6,Third Quartile1,4.299096,5.819304
7,Minimum2,0.000000,0.000000
8,Maximum2,4.598099,5.372570
9,Mean2,1.388105,1.698038


-------------------------------------------------------------------------

#### iv. Select Features

In my opinion different combinations of 3 important features can be considered as the most important ones.

1) Min, Mean and Max - This combination gives us the upper and lower bounds of the distribution, therfore the range as well, and the average value of the distribution. However, they could get affected by outliers.

2) Mean, Standard deviation, Median - This combination focuses on how the information is centred and spread in the distribution, and median is not as affected by outliers as mean, min or max.

--------------------------------------------------------------------------------------------

## 2. ISLR 3.7.4

### (a) Linear Train

The RSS of cubic regression would be smaller than (or equal to) the RSS of linear regression as the higher the order of polynomial, the better it would fit the training data, generally.  

### (b) Linear Test

The RSS for testing data for linear regression would be lower than RSS test of cubic regression as the true relationship is linear, and cubic regression may cause overfitting. Linear regression also has less bias and variance due to the true relationship.

### (c) Not Linear Train

RSS for cubic regression in training data would still be smaller than RSS of linear, as regardless of the true relationship's polynomial degree, cubic can generally fit the model in a more flexible manner than linear.

### (d) Not Linear Testing

There is slight ambiguity here as there is not enough information to judge. If the true relationship is close to linear then RSS of linear would be smaller else RSS of cubic is smaller. However since the true relationship is mentioned as non-linear, it is more likely that cubic would fit the model better, and have low RSS.

-------------------------------------------------------------------------

## REFERENCES


https://stats.stackexchange.com/questions/50807/features-for-time-series-classification


https://stats.stackexchange.com/questions/299302/rss-of-different-models-given-true-relationship-between-x-and-y


https://www.statology.org/bootstrapping-in-python/


https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html



### AI Prompts


Fix code/errors 

How does confidence interval influence important features

Linear vs Cubic RSS